# Results: what happened, and why

Pick a run in **Settings**, then **Run All**. Every figure answers one question; the score numbers come from the course grader. Nothing here changes a submission: it only reads `submissions/best.npy`, the run folder and the logs.

Figures are only shown, not saved. To keep them, set `SAVE_FIGURES = True` and run again: the last cell then writes every figure as a PNG in `<run>/figures/` (git-ignored). Copy the ones you want for the report into a tracked `figures/` folder.

In [ ]:
# Setup: work from the repo root; re-read changed linkopt files before each cell.
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

root = Path.cwd()
while not (root / 'linkopt').is_dir() and root != root.parent:
    root = root.parent
os.chdir(root)

import linkopt  # always import first: pins JAX to the CPU
import matplotlib.pyplot as plt

from linkopt import report
from linkopt.submission import load

print('working in', root)

## Settings

- `RUN`: the run folder to analyse (default: the latest).
- `FACTOR`: the setting to compare (default: whichever one this run varied).
- `LEADERBOARD`: hypervolumes copied from the leaderboard (Problem 1, 2, 3), for scale.
- `SAVE_FIGURES`: also save every figure as a PNG at the end (off by default).

In [ ]:
RUN = report.run_dirs()[-1]          # or e.g. Path('runs/20261002-233526')
FACTOR = None                        # None = whatever this run varied, e.g. 'n_joints'
LEADERBOARD = {                      # from the leaderboard (Problem 1, 2, 3 HV)
    '#1 Shape Grapplers (2026-10-02)': [6.280951, 9.46617, 29.58659],
    '#2 ex-ta (2026-10-02)': [5.514438, 7.914575, 20.837218],
}
SAVE_FIGURES = False

figures = {}  # every figure, by name, for saving at the end
print('run:', RUN)

## 1. How does our submission score?

The grader's numbers for `submissions/best.npy`, the leaderboard's number (the **sum** of the three raw hypervolumes), and how much of each kangaroo's scoring box we cover.

In [ ]:
table = report.score_table()
print(f"{'':32s} {'hypervolume':>11s} {'normalized':>10s} {'box':>6s} {'covered':>8s}")
for r in table['rows']:
    print(f"{r['kangaroo']:32s} {r['hv']:11.3f} {r['hv_norm']:10.3f} {r['box']:6.1f} {r['coverage']:8.1%}")
print(f"\ngrader's overall (average of normalized): {table['overall']:.4f}")
print(f"leaderboard's overall (sum of hypervolumes): {table['leaderboard']:.3f}")
for name, hvs in LEADERBOARD.items():
    print(f'   {name}: {sum(hvs):.3f}  (' + ', '.join(f"{h:.2f}" for h in hvs) + ')')

## 2. Where do our designs sit? (the trade-off)

Each dot is a submitted design; the shaded staircase is the hypervolume (the score). **Left:** zoomed to the designs. **Right:** the whole scoring box, from the ideal point (0, 0) to the limits (the reference point). The ideal point can't be reached, but the unshaded area is score still on the table.

In [ ]:
scored = report.submission_scores(load(report.BEST_PATH))
for t, (designs, F) in scored.items():
    figures[f'tradeoff_k{t + 1}'] = report.plot_trade_off(F, t)
    plt.show()

## 3. What do the best mechanisms look like?

Three designs per kangaroo, all from the front (no other design beats them on both objectives):

- **Closest Fit:** the lowest distance (the bottom-right end of the staircase).
- **Middle:** the middle design when the front is sorted by material.
- **Least Material:** the lowest material (the top-left end).

**How to read each figure.** The title gives the kangaroo, which pick it is, and its distance and material (the two numbers the grader scores).

- **Left, the mechanism** (drawn by the course's `MechanismVisualizer`):
  - black ground symbols = **fixed joints** (pinned to the ground)
  - white circles = **moving joints**; dark bars = **links**
  - the **yellow** bar = the **motor link** (the crank the motor turns)
  - the **solid orange** curve = the path of the **traced joint** (orange dot): the curve that's graded
  - **dashed blue** curves = the paths of the other moving joints
  - the panel title counts the joints and links and names the traced joint.
- **Right, the fit** (the course's `CurveEngine`): **orange** = the traced joint's path, **navy** = the kangaroo, overlaid the way the grader aligns them (shifted and rotated, never resized). *Distance* measures the gap between these two curves; *material* is the total length of the links on the left.

In [ ]:
for t, (designs, F) in scored.items():
    for name, i in report.pick_designs(F).items():
        title = f'{report.KANGAROOS[t]}, {name.title()}'
        figures[f"design_k{t + 1}_{name.replace(' ', '_')}"] = report.plot_design(designs[i], t, title)
        plt.show()

## 4. Designs along the whole front

Section 3 showed three designs; this walks along the front. **Each row is one design**, from the closest fit (top) to the least material (bottom). If a kangaroo's front has more than 6 designs, 6 are picked evenly spaced along it; the title says how many of how many.

**How to read each row:**

- **Left:** the mechanism (same key as section 3).
- **Middle:** its fit over the kangaroo; the panel title gives its distance and material.
- **Right:** the section 2 staircase, with **this design in green**. Reading down the rows, the green dot walks along the staircase from the bottom-right (close fit, lots of material) to the top-left (little material, looser fit).

(The course notebooks' standard figure, plus the staircase.)

In [ ]:
for t, (designs, F) in scored.items():
    figures[f'front_k{t + 1}'] = report.plot_front(designs, F, t, max_rows=6)
    plt.show()

## 5. The DOE: what mattered?

Each job's **own** hypervolume (from `jobs.csv`), grouped by the factor. Kangaroos are separate problems: compare levels **within** a row. **The one rule:** an effect is real only if it's bigger than the spread between seeds.

Every number here is **one job's own hypervolume**: the score its designs would get alone, before pooling.

- **Heatmap:** colour = the median job's share of the scoring box; text = the median and [IQR], and how many seeds found at least one design within the limits (the rest score 0). With many zeros, the median and brackets can all be 0 even though a few seeds scored: the box plots show those as dots.
  - **IQR** = interquartile range, the 25th to 75th percentile: the middle half of the seeds. Narrow = the seeds agree; two cells whose IQRs don't overlap differ by more than the seed noise.
- **Box plots:** the full spread over the seeds. Box = IQR (the same brackets), line = median, whiskers = the rest, dots = outliers.

In [ ]:
rows = report.read_jobs(RUN)
varied = report.factors_varied(rows)
factor = FACTOR or (varied[0] if varied else None)
print(f'{len(rows)} GA jobs; settings varied: {varied or "none"}')
if factor is None:
    print('This run used one value of every setting, so there is nothing to compare.')
else:
    figures['doe_heatmap'] = report.plot_doe_heatmap(rows, factor)
    plt.show()
    figures['doe_boxes'] = report.plot_doe_boxes(rows, factor)
    plt.show()

## 6. Does refinement help?

Every job runs both stages: the GA, then gradient refinement of the GA's designs. Each dot is **one job's hypervolume, measured twice**: as its GA finishes, before refining (x), and after refining (y). The job keeps both the original and the refined version of each design, so a dot can never fall below the grey diagonal; its height above it is what the local search added on top of the global search.

In [ ]:
if factor is not None:
    figures['ga_vs_refined'] = report.plot_ga_vs_refined(rows, factor)
    plt.show()

## 7. Do more seeds keep paying?

The hypervolume of all designs from the first *k* seeds, pooled. Where a line flattens, more replicates of that setting stop adding score. (Scores every job's designs once: takes ~10-20 s for a big run.)

In [ ]:
if factor is not None:
    curve = report.seeds_curve(rows, report.job_F(RUN), factor)
    figures['seeds_curve'] = report.plot_seeds_curve(curve, factor)
    plt.show()

## 8. Where did the designs in our submission come from?

Which kinds of jobs produced the designs now in `best.npy`, searching every run folder on this computer. "Not in This Computer's Runs": the starter baseline, or a teammate's merged file.

In [ ]:
prov = report.provenance()
for t, counts in prov.items():
    print(report.KANGAROOS[t], dict(counts.most_common()))
figures['provenance'] = report.plot_provenance(prov)
plt.show()

## Save the figures

In [ ]:
if SAVE_FIGURES:
    for path in report.save_figures(figures, RUN / 'figures'):
        print(path)
else:
    print('SAVE_FIGURES is off: nothing saved')